In [1]:
"""
realtor_scraper.py
===================

Scrapes property listings from REALTOR.ca's own search API (the same
endpoint the website's JavaScript calls when you browse listings or change
pages) and saves the results as JSON.

HOW THIS WAS BUILT
-------------------
REALTOR.ca renders its listing pages client-side. The actual data comes
from a POST request to:

    https://api2.realtor.ca/Listing.svc/AsyncPropertySearch_Post

This was confirmed by watching the site's own network traffic while
paginating through https://www.realtor.ca/ab/calgary/real-estate. Because
this hits REALTOR.ca's internal API directly (instead of parsing rendered
HTML), it's much less fragile than a classic HTML scraper -- but it also
means it can break if REALTOR.ca changes that API without notice.

IMPORTANT LIMITS / THINGS TO KNOW
----------------------------------
1. REALTOR.ca caps how many listings you can page through for any single
   search/sort combination -- in testing, the API reported a hard cap of
   ~600 records ("MaxRecords") even though the Calgary search matched
   ~6,900+ listings total. This is a deliberate limit on their end, not a
   bug here. If you need more coverage, run this script multiple times
   with different filters (price bands, property type, etc. -- see
   `build_payload()`) and merge/dedupe the results by `mls_number`.
2. This calls a private/undocumented API that isn't meant for third-party
   use. REALTOR.ca's Terms of Use restrict automated scraping and
   redistributing MLS data commercially -- this script is intended for
   personal, non-commercial, rate-limited use (e.g. tracking listings
   you're personally interested in). Please review realtor.ca's Terms of
   Use yourself before relying on this, and don't hammer their servers --
   the default delay between requests is intentionally conservative.
3. REALTOR.ca may have bot-detection in front of this API. This script
   sends browser-like headers and warms up a session cookie first, which
   worked as of the time this was written, but if you start getting
   403/blocked responses, that's REALTOR.ca's anti-bot layer kicking in --
   slow down the --delay, or reduce request volume.

USAGE
-----
    python3 realtor_scraper.py
    python3 realtor_scraper.py --pages 20 --delay 2 --output calgary.json
    python3 realtor_scraper.py --geo-id g30_c3nfkdtg --records-per-page 20

Finding a GEO-ID for a different city/area:
    1. Open https://www.realtor.ca in a normal browser and search the
       area you want.
    2. Open DevTools -> Network tab, filter for "AsyncPropertySearch_Post".
    3. Look at the request's form body -- copy the "GeoIds" value.
    4. Pass it here with --geo-id.

Output is a single JSON file: a list of listing objects (see
`parse_listing()` for the exact fields), plus a small metadata block.
"""

import argparse
import json
import re
import sys
import time
from datetime import datetime, timezone

import requests

SEARCH_URL = "https://api2.realtor.ca/Listing.svc/AsyncPropertySearch_Post"
WARMUP_URL = "https://www.realtor.ca/ab/calgary/real-estate"

DEFAULT_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Accept-Language": "en-CA,en;q=0.9",
    "X-Requested-With": "XMLHttpRequest",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "Referer": WARMUP_URL,
    "Origin": "https://www.realtor.ca",
}


def build_payload(geo_id, page, records_per_page, transaction_type_id,
                   property_type_group_id, sort):
    """Build the form-encoded body REALTOR.ca's API expects.

    Field meanings, as observed from the live site (Calgary, "For Sale",
    residential, default "Newest" sort):
      GeoIds               -- area identifier (see module docstring)
      TransactionTypeId    -- 2 = For Sale on the page this was captured
                               from; REALTOR.ca uses a different value for
                               rentals. If you need rentals, browse
                               realtor.ca/.../rentals and re-capture this
                               value from the Network tab.
      PropertyTypeGroupID  -- 1 = Residential
      PropertySearchTypeId -- 1 (constant on the page this was captured)
      Sort                 -- "6-D" = Newest first (the page's default).
                               Other sort orders exist (price, etc.) but
                               weren't captured/verified here -- check the
                               Network tab if you want to add one.
    """
    return {
        "CurrentPage": page,
        "Sort": sort,
        "GeoIds": geo_id,
        "PropertyTypeGroupID": property_type_group_id,
        "TransactionTypeId": transaction_type_id,
        "PropertySearchTypeId": 1,
        "Currency": "CAD",
        "IncludeHiddenListings": "false",
        "RecordsPerPage": records_per_page,
        "ApplicationId": 1,
        "CultureId": 1,
        "Version": "7.0",
    }


def _num(text):
    """Best-effort: pull a number out of strings like '$685,000' or '1341 sqft'."""
    if not text:
        return None
    match = re.search(r"[\d,]+(\.\d+)?", str(text))
    if not match:
        return None
    try:
        return float(match.group(0).replace(",", ""))
    except ValueError:
        return None


def parse_listing(item):
    """Flatten one raw API listing object into a clean, JSON-friendly dict."""
    prop = item.get("Property") or {}
    building = item.get("Building") or {}
    address = prop.get("Address") or {}
    photos = prop.get("Photo") or []
    individuals = item.get("Individual") or []
    agent = individuals[0] if individuals else {}
    org = (agent.get("Organization") or {}) if agent else {}

    address_text = address.get("AddressText", "") or ""
    # AddressText looks like "365 Sunmills Drive SE|Calgary, Alberta T2X2T5"
    street, _, city_prov_postal = address_text.partition("|")

    price_raw = prop.get("Price")

    return {
        "mls_number": item.get("MlsNumber"),
        "listing_url": (
            "https://www.realtor.ca" + item["RelativeDetailsURL"]
            if item.get("RelativeDetailsURL") else None
        ),
        "price_raw": price_raw,
        "price": _num(price_raw),
        "property_type": prop.get("Type"),
        "address": street.strip(),
        "city_province_postal": city_prov_postal.strip(),
        "latitude": address.get("Latitude"),
        "longitude": address.get("Longitude"),
        "bedrooms": building.get("Bedrooms"),
        "bathrooms": building.get("BathroomTotal"),
        "half_bathrooms": building.get("HalfBathTotal"),
        "size_interior": building.get("SizeInterior"),
        "stories": building.get("StoriesTotal"),
        "building_type": building.get("Type"),
        "parking_spaces_total": prop.get("ParkingSpaceTotal"),
        "description": item.get("PublicRemarks"),
        "photo_url": photos[0].get("MedResPath") if photos else None,
        "listed_date_utc": item.get("InsertedDateUTC"),
        "days_on_market": item.get("TimeOnRealtor"),
        "agent_name": agent.get("Name"),
        "brokerage_name": org.get("Name"),
        "brokerage_phone": (
            org.get("Phones", [{}])[0].get("PhoneNumber")
            if org.get("Phones") else None
        ),
    }


def scrape(geo_id, max_pages, records_per_page, delay, transaction_type_id,
           property_type_group_id, sort):
    session = requests.Session()
    session.headers.update(DEFAULT_HEADERS)

    # Warm up: fetch the human-facing page first so the session picks up
    # any cookies REALTOR.ca sets before we start hitting the API.
    try:
        session.get(WARMUP_URL, timeout=20)
    except requests.RequestException as exc:
        print(f"Warning: warm-up request failed ({exc}); continuing anyway.",
              file=sys.stderr)

    all_listings = []
    seen_mls = set()
    total_records = None
    max_records_cap = None

    page = 1
    while max_pages is None or page <= max_pages:
        payload = build_payload(
            geo_id, page, records_per_page, transaction_type_id,
            property_type_group_id, sort,
        )

        for attempt in range(3):
            try:
                resp = session.post(SEARCH_URL, data=payload, timeout=20)
                break
            except requests.RequestException as exc:
                print(f"Page {page}: request error ({exc}), "
                      f"retry {attempt + 1}/3...", file=sys.stderr)
                time.sleep(2 * (attempt + 1))
        else:
            print(f"Page {page}: giving up after 3 failed attempts.",
                  file=sys.stderr)
            break

        if resp.status_code != 200:
            print(f"Page {page}: HTTP {resp.status_code} -- stopping. "
                  f"(REALTOR.ca may be rate-limiting/blocking this request.)",
                  file=sys.stderr)
            break

        try:
            data = resp.json()
        except ValueError:
            print(f"Page {page}: response wasn't valid JSON -- stopping.",
                  file=sys.stderr)
            break

        # ErrorCode has appeared in two shapes from this API: a bare value
        # (None / 200 / "200") on older responses, and a nested object like
        # {'Id': 200, 'Description': 'Success - OK', ...} on newer ones.
        # Normalize to just the numeric/None code before checking it, so a
        # successful nested response doesn't get misread as an error.
        error_code = data.get("ErrorCode")
        error_id = error_code.get("Id") if isinstance(error_code, dict) else error_code

        if error_id not in (None, "200", 200):
            print(f"Page {page}: API returned ErrorCode={error_code} "
                  f"-- stopping.", file=sys.stderr)
            break

        paging = data.get("Paging") or {}
        total_records = paging.get("TotalRecords", total_records)
        max_records_cap = paging.get("MaxRecords", max_records_cap)

        results = data.get("Results") or []
        if not results:
            print(f"Page {page}: no more results.", file=sys.stderr)
            break

        new_count = 0
        for raw in results:
            parsed = parse_listing(raw)
            mls = parsed.get("mls_number")
            if mls and mls in seen_mls:
                continue
            if mls:
                seen_mls.add(mls)
            all_listings.append(parsed)
            new_count += 1

        print(f"Page {page}: got {len(results)} listings "
              f"({new_count} new, {len(all_listings)} total so far).",
              file=sys.stderr)

        records_showing = paging.get("RecordsShowing")
        if max_records_cap and records_showing and records_showing >= max_records_cap:
            print("Reached REALTOR.ca's MaxRecords cap for this search/sort "
                  "-- stopping. See the module docstring for how to get more.",
                  file=sys.stderr)
            break

        total_pages = paging.get("TotalPages")
        if total_pages and page >= total_pages:
            break

        page += 1
        time.sleep(delay)

    return {
        "listings": all_listings,
        "meta": {
            "scraped_at_utc": datetime.now(timezone.utc).isoformat(),
            "source": WARMUP_URL,
            "geo_id": geo_id,
            "sort": sort,
            "listings_scraped": len(all_listings),
            "total_records_matching_search": total_records,
            "realtor_ca_page_cap": max_records_cap,
        },
    }


def main():
    parser = argparse.ArgumentParser(
        description="Scrape REALTOR.ca listings into a JSON file.")
    parser.add_argument("--geo-id", default="g30_c3nfkdtg",
                         help="REALTOR.ca GeoIds value (default: Calgary, AB).")
    parser.add_argument("--pages", type=int, default=None,
                         help="Max number of pages to fetch (default: until "
                              "REALTOR.ca's cap or results run out).")
    parser.add_argument("--records-per-page", type=int, default=50,
                         help="Listings per request (default: 50).")
    parser.add_argument("--delay", type=float, default=1.5,
                         help="Seconds to wait between requests (default: 1.5).")
    parser.add_argument("--transaction-type-id", type=int, default=2,
                         help="2 = For Sale, as captured from the site "
                              "(default: 2).")
    parser.add_argument("--property-type-group-id", type=int, default=1,
                         help="1 = Residential (default: 1).")
    parser.add_argument("--sort", default="6-D",
                         help="Sort order code (default: '6-D' = Newest).")
    parser.add_argument("--output", default="realtor_listings.json",
                         help="Output JSON file path.")


    args, _unknown_args = parser.parse_known_args()

    result = scrape(
        geo_id=args.geo_id,
        max_pages=args.pages,
        records_per_page=args.records_per_page,
        delay=args.delay,
        transaction_type_id=args.transaction_type_id,
        property_type_group_id=args.property_type_group_id,
        sort=args.sort,
    )

    with open(args.output, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f"\nSaved {len(result['listings'])} listings to {args.output}")
    print(f"(REALTOR.ca reported {result['meta']['total_records_matching_search']} "
          f"total matching listings; it caps browsable results at "
          f"{result['meta']['realtor_ca_page_cap']} per search/sort.)")


if __name__ == "__main__":
    main()


Saved 50 listings to realtor_listings.json
(REALTOR.ca reported 6899 total matching listings; it caps browsable results at 600 per search/sort.)


Page 1: got 50 listings (50 new, 50 total so far).
Reached REALTOR.ca's MaxRecords cap for this search/sort -- stopping. See the module docstring for how to get more.


In [2]:
#!/usr/bin/env python3
"""
realtor_scraper.py
===================

Scrapes property listings from REALTOR.ca's own search API (the same
endpoint the website's JavaScript calls when you browse listings or change
pages) and saves the results as JSON.

HOW THIS WAS BUILT
-------------------
REALTOR.ca renders its listing pages client-side. The actual data comes
from a POST request to:

    https://api2.realtor.ca/Listing.svc/AsyncPropertySearch_Post

This was confirmed by watching the site's own network traffic while
paginating through https://www.realtor.ca/ab/calgary/real-estate. Because
this hits REALTOR.ca's internal API directly (instead of parsing rendered
HTML), it's much less fragile than a classic HTML scraper -- but it also
means it can break if REALTOR.ca changes that API without notice.

IMPORTANT LIMITS / THINGS TO KNOW
----------------------------------
1. REALTOR.ca caps how many listings you can page through for any single
   search/sort combination -- in testing, the API reported a hard cap of
   ~600 records ("MaxRecords") even though the Calgary search matched
   ~6,900+ listings total. This is a deliberate limit on their end, not a
   bug here. If you need more coverage, run this script multiple times
   with different filters (price bands, property type, etc. -- see
   `build_payload()`) and merge/dedupe the results by `mls_number`.
2. This calls a private/undocumented API that isn't meant for third-party
   use. REALTOR.ca's Terms of Use restrict automated scraping and
   redistributing MLS data commercially -- this script is intended for
   personal, non-commercial, rate-limited use (e.g. tracking listings
   you're personally interested in). Please review realtor.ca's Terms of
   Use yourself before relying on this, and don't hammer their servers --
   the default delay between requests is intentionally conservative.
3. REALTOR.ca may have bot-detection in front of this API. This script
   sends browser-like headers and warms up a session cookie first, which
   worked as of the time this was written, but if you start getting
   403/blocked responses, that's REALTOR.ca's anti-bot layer kicking in --
   slow down the --delay, or reduce request volume.

USAGE
-----
    python3 realtor_scraper.py
    python3 realtor_scraper.py --pages 20 --delay 2 --output calgary.json
    python3 realtor_scraper.py --geo-id g30_c3nfkdtg --records-per-page 20

Finding a GEO-ID for a different city/area:
    1. Open https://www.realtor.ca in a normal browser and search the
       area you want.
    2. Open DevTools -> Network tab, filter for "AsyncPropertySearch_Post".
    3. Look at the request's form body -- copy the "GeoIds" value.
    4. Pass it here with --geo-id.

Output is a single JSON file: a list of listing objects (see
`parse_listing()` for the exact fields), plus a small metadata block.
"""

import argparse
import csv
import json
import os
import re
import sys
import time
from datetime import datetime, timezone

import requests

SEARCH_URL = "https://api2.realtor.ca/Listing.svc/AsyncPropertySearch_Post"
WARMUP_URL = "https://www.realtor.ca/ab/calgary/real-estate"

DEFAULT_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Accept-Language": "en-CA,en;q=0.9",
    "X-Requested-With": "XMLHttpRequest",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "Referer": WARMUP_URL,
    "Origin": "https://www.realtor.ca",
}


def build_payload(geo_id, page, records_per_page, transaction_type_id,
                   property_type_group_id, sort):
    """Build the form-encoded body REALTOR.ca's API expects.

    Field meanings, as observed from the live site (Calgary, "For Sale",
    residential, default "Newest" sort):
      GeoIds               -- area identifier (see module docstring)
      TransactionTypeId    -- 2 = For Sale on the page this was captured
                               from; REALTOR.ca uses a different value for
                               rentals. If you need rentals, browse
                               realtor.ca/.../rentals and re-capture this
                               value from the Network tab.
      PropertyTypeGroupID  -- 1 = Residential
      PropertySearchTypeId -- 1 (constant on the page this was captured)
      Sort                 -- "6-D" = Newest first (the page's default).
                               Other sort orders exist (price, etc.) but
                               weren't captured/verified here -- check the
                               Network tab if you want to add one.
    """
    return {
        "CurrentPage": page,
        "Sort": sort,
        "GeoIds": geo_id,
        "PropertyTypeGroupID": property_type_group_id,
        "TransactionTypeId": transaction_type_id,
        "PropertySearchTypeId": 1,
        "Currency": "CAD",
        "IncludeHiddenListings": "false",
        "RecordsPerPage": records_per_page,
        "ApplicationId": 1,
        "CultureId": 1,
        "Version": "7.0",
    }


def _num(text):
    """Best-effort: pull a number out of strings like '$685,000' or '1341 sqft'."""
    if not text:
        return None
    match = re.search(r"[\d,]+(\.\d+)?", str(text))
    if not match:
        return None
    try:
        return float(match.group(0).replace(",", ""))
    except ValueError:
        return None


def parse_listing(item):
    """Flatten one raw API listing object into a clean, JSON-friendly dict."""
    prop = item.get("Property") or {}
    building = item.get("Building") or {}
    address = prop.get("Address") or {}
    photos = prop.get("Photo") or []
    individuals = item.get("Individual") or []
    agent = individuals[0] if individuals else {}
    org = (agent.get("Organization") or {}) if agent else {}

    address_text = address.get("AddressText", "") or ""
    # AddressText looks like "365 Sunmills Drive SE|Calgary, Alberta T2X2T5"
    street, _, city_prov_postal = address_text.partition("|")

    price_raw = prop.get("Price")

    return {
        "mls_number": item.get("MlsNumber"),
        "listing_url": (
            "https://www.realtor.ca" + item["RelativeDetailsURL"]
            if item.get("RelativeDetailsURL") else None
        ),
        "price_raw": price_raw,
        "price": _num(price_raw),
        "property_type": prop.get("Type"),
        "address": street.strip(),
        "city_province_postal": city_prov_postal.strip(),
        "latitude": address.get("Latitude"),
        "longitude": address.get("Longitude"),
        "bedrooms": building.get("Bedrooms"),
        "bathrooms": building.get("BathroomTotal"),
        "half_bathrooms": building.get("HalfBathTotal"),
        "size_interior": building.get("SizeInterior"),
        "stories": building.get("StoriesTotal"),
        "building_type": building.get("Type"),
        "parking_spaces_total": prop.get("ParkingSpaceTotal"),
        "description": item.get("PublicRemarks"),
        "photo_url": photos[0].get("MedResPath") if photos else None,
        "listed_date_utc": item.get("InsertedDateUTC"),
        "days_on_market": item.get("TimeOnRealtor"),
        "agent_name": agent.get("Name"),
        "brokerage_name": org.get("Name"),
        "brokerage_phone": (
            org.get("Phones", [{}])[0].get("PhoneNumber")
            if org.get("Phones") else None
        ),
    }


# Fixed column order for the CSV export -- matches parse_listing()'s keys.
# Kept as an explicit list (rather than reading it off the first result) so
# the CSV always has the same header row even when 0 listings were scraped.
LISTING_FIELDNAMES = [
    "mls_number", "listing_url", "price_raw", "price", "property_type",
    "address", "city_province_postal", "latitude", "longitude",
    "bedrooms", "bathrooms", "half_bathrooms", "size_interior", "stories",
    "building_type", "parking_spaces_total", "description", "photo_url",
    "listed_date_utc", "days_on_market", "agent_name", "brokerage_name",
    "brokerage_phone",
]


def save_csv(listings, path):
    """Write the parsed listings to an Excel-friendly CSV file.

    Uses utf-8-sig (adds a BOM) so Excel correctly displays non-ASCII
    characters (e.g. accented street/city names) instead of mangling them.
    """
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=LISTING_FIELDNAMES,
                                 extrasaction="ignore")
        writer.writeheader()
        for listing in listings:
            writer.writerow(listing)


def scrape(geo_id, max_pages, records_per_page, delay, transaction_type_id,
           property_type_group_id, sort):
    session = requests.Session()
    session.headers.update(DEFAULT_HEADERS)

    # Warm up: fetch the human-facing page first so the session picks up
    # any cookies REALTOR.ca sets before we start hitting the API.
    try:
        session.get(WARMUP_URL, timeout=20)
    except requests.RequestException as exc:
        print(f"Warning: warm-up request failed ({exc}); continuing anyway.",
              file=sys.stderr)

    all_listings = []
    seen_mls = set()
    total_records = None
    max_records_cap = None

    page = 1
    while max_pages is None or page <= max_pages:
        payload = build_payload(
            geo_id, page, records_per_page, transaction_type_id,
            property_type_group_id, sort,
        )

        for attempt in range(3):
            try:
                resp = session.post(SEARCH_URL, data=payload, timeout=20)
                break
            except requests.RequestException as exc:
                print(f"Page {page}: request error ({exc}), "
                      f"retry {attempt + 1}/3...", file=sys.stderr)
                time.sleep(2 * (attempt + 1))
        else:
            print(f"Page {page}: giving up after 3 failed attempts.",
                  file=sys.stderr)
            break

        if resp.status_code != 200:
            print(f"Page {page}: HTTP {resp.status_code} -- stopping. "
                  f"(REALTOR.ca may be rate-limiting/blocking this request.)",
                  file=sys.stderr)
            break

        try:
            data = resp.json()
        except ValueError:
            print(f"Page {page}: response wasn't valid JSON -- stopping.",
                  file=sys.stderr)
            break

        # ErrorCode has appeared in two shapes from this API: a bare value
        # (None / 200 / "200") on older responses, and a nested object like
        # {'Id': 200, 'Description': 'Success - OK', ...} on newer ones.
        # Normalize to just the numeric/None code before checking it, so a
        # successful nested response doesn't get misread as an error.
        error_code = data.get("ErrorCode")
        error_id = error_code.get("Id") if isinstance(error_code, dict) else error_code

        if error_id not in (None, "200", 200):
            print(f"Page {page}: API returned ErrorCode={error_code} "
                  f"-- stopping.", file=sys.stderr)
            break

        paging = data.get("Paging") or {}
        total_records = paging.get("TotalRecords", total_records)
        max_records_cap = paging.get("MaxRecords", max_records_cap)

        results = data.get("Results") or []
        if not results:
            print(f"Page {page}: no more results.", file=sys.stderr)
            break

        new_count = 0
        for raw in results:
            parsed = parse_listing(raw)
            mls = parsed.get("mls_number")
            if mls and mls in seen_mls:
                continue
            if mls:
                seen_mls.add(mls)
            all_listings.append(parsed)
            new_count += 1

        print(f"Page {page}: got {len(results)} listings "
              f"({new_count} new, {len(all_listings)} total so far).",
              file=sys.stderr)

        records_showing = paging.get("RecordsShowing")
        if max_records_cap and records_showing and records_showing >= max_records_cap:
            print("Reached REALTOR.ca's MaxRecords cap for this search/sort "
                  "-- stopping. See the module docstring for how to get more.",
                  file=sys.stderr)
            break

        total_pages = paging.get("TotalPages")
        if total_pages and page >= total_pages:
            break

        page += 1
        time.sleep(delay)

    return {
        "listings": all_listings,
        "meta": {
            "scraped_at_utc": datetime.now(timezone.utc).isoformat(),
            "source": WARMUP_URL,
            "geo_id": geo_id,
            "sort": sort,
            "listings_scraped": len(all_listings),
            "total_records_matching_search": total_records,
            "realtor_ca_page_cap": max_records_cap,
        },
    }


def main():
    parser = argparse.ArgumentParser(
        description="Scrape REALTOR.ca listings into a JSON file.")
    parser.add_argument("--geo-id", default="g30_c3nfkdtg",
                         help="REALTOR.ca GeoIds value (default: Calgary, AB).")
    parser.add_argument("--pages", type=int, default=None,
                         help="Max number of pages to fetch (default: until "
                              "REALTOR.ca's cap or results run out).")
    parser.add_argument("--records-per-page", type=int, default=50,
                         help="Listings per request (default: 50).")
    parser.add_argument("--delay", type=float, default=1.5,
                         help="Seconds to wait between requests (default: 1.5).")
    parser.add_argument("--transaction-type-id", type=int, default=2,
                         help="2 = For Sale, as captured from the site "
                              "(default: 2).")
    parser.add_argument("--property-type-group-id", type=int, default=1,
                         help="1 = Residential (default: 1).")
    parser.add_argument("--sort", default="6-D",
                         help="Sort order code (default: '6-D' = Newest).")
    parser.add_argument("--output", default="realtor_listings.json",
                         help="Output JSON file path.")
    parser.add_argument("--csv-output", default=None,
                         help="Also save listings as CSV at this path "
                              "(default: same name as --output with a "
                              ".csv extension). Pass 'none' to skip CSV "
                              "export.")

    # parse_known_args() (instead of parse_args()) so this still works when
    # run inside Jupyter: Jupyter launches the kernel as
    # "ipykernel_launcher.py -f <connection-file>.json", and that "-f ..."
    # ends up in sys.argv. parse_args() would choke on it as an unrecognized
    # argument ("unrecognized arguments: -f ...json"); parse_known_args()
    # just ignores anything it doesn't recognize and falls back to the
    # defaults above. Works identically from a normal command line too.
    args, _unknown_args = parser.parse_known_args()

    result = scrape(
        geo_id=args.geo_id,
        max_pages=args.pages,
        records_per_page=args.records_per_page,
        delay=args.delay,
        transaction_type_id=args.transaction_type_id,
        property_type_group_id=args.property_type_group_id,
        sort=args.sort,
    )

    # Resolve to an absolute path before writing. A bare relative filename
    # (the default) already lands in the current working directory -- which
    # in Jupyter is normally the notebook's own folder ("project directory")
    # -- but resolving it explicitly means the path printed below tells you
    # exactly where to look, instead of leaving you to guess.
    output_path = os.path.abspath(os.path.expanduser(args.output))
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f"\nSaved {len(result['listings'])} listings to {output_path}")

    if args.csv_output != "none":
        csv_path = args.csv_output
        if csv_path is None:
            base, _ext = os.path.splitext(args.output)
            csv_path = base + ".csv"
        csv_path = os.path.abspath(os.path.expanduser(csv_path))
        os.makedirs(os.path.dirname(csv_path) or ".", exist_ok=True)
        save_csv(result["listings"], csv_path)
        print(f"Saved {len(result['listings'])} listings to {csv_path} (CSV)")

    print(f"(REALTOR.ca reported {result['meta']['total_records_matching_search']} "
          f"total matching listings; it caps browsable results at "
          f"{result['meta']['realtor_ca_page_cap']} per search/sort.)")


if __name__ == "__main__":
    main()


Saved 50 listings to c:\Users\Sunanda2392\AppData\Local\Programs\Microsoft VS Code\realtor_listings.json
Saved 50 listings to c:\Users\Sunanda2392\AppData\Local\Programs\Microsoft VS Code\realtor_listings.csv (CSV)
(REALTOR.ca reported 6899 total matching listings; it caps browsable results at 600 per search/sort.)


Page 1: got 50 listings (50 new, 50 total so far).
Reached REALTOR.ca's MaxRecords cap for this search/sort -- stopping. See the module docstring for how to get more.


In [3]:
#!/usr/bin/env python3
"""
realtor_scraper.py
===================

Scrapes property listings from REALTOR.ca's own search API (the same
endpoint the website's JavaScript calls when you browse listings or change
pages) and saves the results as JSON.

HOW THIS WAS BUILT
-------------------
REALTOR.ca renders its listing pages client-side. The actual data comes
from a POST request to:

    https://api2.realtor.ca/Listing.svc/AsyncPropertySearch_Post

This was confirmed by watching the site's own network traffic while
paginating through https://www.realtor.ca/ab/calgary/real-estate. Because
this hits REALTOR.ca's internal API directly (instead of parsing rendered
HTML), it's much less fragile than a classic HTML scraper -- but it also
means it can break if REALTOR.ca changes that API without notice.

IMPORTANT LIMITS / THINGS TO KNOW
----------------------------------
1. REALTOR.ca caps how many listings you can page through for any single
   search/sort combination -- in testing, the API reported a hard cap of
   ~600 records ("MaxRecords") even though the Calgary search matched
   ~6,900+ listings total. This is a deliberate limit on their end, not a
   bug here. To work around it, this script automatically splits the
   search into price bands (using PriceMin/PriceMax) whenever the total
   exceeds that cap, scrapes each band separately, and merges/dedupes the
   results by `mls_number` -- see `_split_price_bands()` and `scrape()`.
   This relies on REALTOR.ca actually honoring PriceMin/PriceMax, which
   the script sanity-checks before trusting it (see
   `_price_filter_supported()`); if that check fails it prints a warning
   and falls back to the single capped pass instead of silently producing
   incomplete/duplicate data. Pass --no-price-split to disable this and
   always take the single capped pass.
2. This calls a private/undocumented API that isn't meant for third-party
   use. REALTOR.ca's Terms of Use restrict automated scraping and
   redistributing MLS data commercially -- this script is intended for
   personal, non-commercial, rate-limited use (e.g. tracking listings
   you're personally interested in). Please review realtor.ca's Terms of
   Use yourself before relying on this, and don't hammer their servers --
   the default delay between requests is intentionally conservative.
3. REALTOR.ca may have bot-detection in front of this API. This script
   sends browser-like headers and warms up a session cookie first, which
   worked as of the time this was written, but if you start getting
   403/blocked responses, that's REALTOR.ca's anti-bot layer kicking in --
   slow down the --delay, or reduce request volume.

USAGE
-----
    python3 realtor_scraper.py
    python3 realtor_scraper.py --pages 20 --delay 2 --output calgary.json
    python3 realtor_scraper.py --geo-id g30_c3nfkdtg --records-per-page 20
    python3 realtor_scraper.py --no-price-split   # disable price-band splitting

Finding a GEO-ID for a different city/area:
    1. Open https://www.realtor.ca in a normal browser and search the
       area you want.
    2. Open DevTools -> Network tab, filter for "AsyncPropertySearch_Post".
    3. Look at the request's form body -- copy the "GeoIds" value.
    4. Pass it here with --geo-id.

Output is a single JSON file: a list of listing objects (see
`parse_listing()` for the exact fields), plus a small metadata block.
"""

import argparse
import csv
import json
import os
import re
import sys
import time
from datetime import datetime, timezone

import requests

SEARCH_URL = "https://api2.realtor.ca/Listing.svc/AsyncPropertySearch_Post"
WARMUP_URL = "https://www.realtor.ca/ab/calgary/real-estate"

# Where output files land by default when --output / --csv-output aren't
# given explicitly. Created automatically if it doesn't exist yet (see
# main()). Override per-run with --output/--csv-output if you ever need a
# different location.
DEFAULT_OUTPUT_DIR = r"D:\backup\Downloads\RealtorScrapperProject"

DEFAULT_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Accept-Language": "en-CA,en;q=0.9",
    "X-Requested-With": "XMLHttpRequest",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "Referer": WARMUP_URL,
    "Origin": "https://www.realtor.ca",
}


def build_payload(geo_id, page, records_per_page, transaction_type_id,
                   property_type_group_id, sort, price_min=None,
                   price_max=None):
    """Build the form-encoded body REALTOR.ca's API expects.

    Field meanings, as observed from the live site (Calgary, "For Sale",
    residential, default "Newest" sort):
      GeoIds               -- area identifier (see module docstring)
      TransactionTypeId    -- 2 = For Sale on the page this was captured
                               from; REALTOR.ca uses a different value for
                               rentals. If you need rentals, browse
                               realtor.ca/.../rentals and re-capture this
                               value from the Network tab.
      PropertyTypeGroupID  -- 1 = Residential
      PropertySearchTypeId -- 1 (constant on the page this was captured)
      Sort                 -- "6-D" = Newest first (the page's default).
                               Other sort orders exist (price, etc.) but
                               weren't captured/verified here -- check the
                               Network tab if you want to add one.
      PriceMin / PriceMax  -- optional price-range filter, used by
                               _split_price_bands() to work around
                               REALTOR.ca's per-search results cap. Only
                               included in the request when given (None =
                               no bound on that side). NOTE: these field
                               names are the commonly-documented ones for
                               this endpoint but weren't independently
                               re-verified against the live API while this
                               was written -- scrape() sanity-checks that
                               they're actually being honored before
                               relying on them (see
                               _price_filter_supported()).
    """
    payload = {
        "CurrentPage": page,
        "Sort": sort,
        "GeoIds": geo_id,
        "PropertyTypeGroupID": property_type_group_id,
        "TransactionTypeId": transaction_type_id,
        "PropertySearchTypeId": 1,
        "Currency": "CAD",
        "IncludeHiddenListings": "false",
        "RecordsPerPage": records_per_page,
        "ApplicationId": 1,
        "CultureId": 1,
        "Version": "7.0",
    }
    if price_min is not None:
        payload["PriceMin"] = price_min
    if price_max is not None:
        payload["PriceMax"] = price_max
    return payload


def _num(text):
    """Best-effort: pull a number out of strings like '$685,000' or '1341 sqft'."""
    if not text:
        return None
    match = re.search(r"[\d,]+(\.\d+)?", str(text))
    if not match:
        return None
    try:
        return float(match.group(0).replace(",", ""))
    except ValueError:
        return None


def parse_listing(item):
    """Flatten one raw API listing object into a clean, JSON-friendly dict."""
    prop = item.get("Property") or {}
    building = item.get("Building") or {}
    address = prop.get("Address") or {}
    photos = prop.get("Photo") or []
    individuals = item.get("Individual") or []
    agent = individuals[0] if individuals else {}
    org = (agent.get("Organization") or {}) if agent else {}

    address_text = address.get("AddressText", "") or ""
    # AddressText looks like "365 Sunmills Drive SE|Calgary, Alberta T2X2T5"
    street, _, city_prov_postal = address_text.partition("|")

    price_raw = prop.get("Price")

    return {
        "mls_number": item.get("MlsNumber"),
        "listing_url": (
            "https://www.realtor.ca" + item["RelativeDetailsURL"]
            if item.get("RelativeDetailsURL") else None
        ),
        "price_raw": price_raw,
        "price": _num(price_raw),
        "property_type": prop.get("Type"),
        "address": street.strip(),
        "city_province_postal": city_prov_postal.strip(),
        "latitude": address.get("Latitude"),
        "longitude": address.get("Longitude"),
        "bedrooms": building.get("Bedrooms"),
        "bathrooms": building.get("BathroomTotal"),
        "half_bathrooms": building.get("HalfBathTotal"),
        "size_interior": building.get("SizeInterior"),
        "stories": building.get("StoriesTotal"),
        "building_type": building.get("Type"),
        "parking_spaces_total": prop.get("ParkingSpaceTotal"),
        "description": item.get("PublicRemarks"),
        "photo_url": photos[0].get("MedResPath") if photos else None,
        "listed_date_utc": item.get("InsertedDateUTC"),
        "days_on_market": item.get("TimeOnRealtor"),
        "agent_name": agent.get("Name"),
        "brokerage_name": org.get("Name"),
        "brokerage_phone": (
            org.get("Phones", [{}])[0].get("PhoneNumber")
            if org.get("Phones") else None
        ),
    }


# Fixed column order for the CSV export -- matches parse_listing()'s keys.
# Kept as an explicit list (rather than reading it off the first result) so
# the CSV always has the same header row even when 0 listings were scraped.
LISTING_FIELDNAMES = [
    "mls_number", "listing_url", "price_raw", "price", "property_type",
    "address", "city_province_postal", "latitude", "longitude",
    "bedrooms", "bathrooms", "half_bathrooms", "size_interior", "stories",
    "building_type", "parking_spaces_total", "description", "photo_url",
    "listed_date_utc", "days_on_market", "agent_name", "brokerage_name",
    "brokerage_phone",
]


def save_csv(listings, path):
    """Write the parsed listings to an Excel-friendly CSV file.

    Uses utf-8-sig (adds a BOM) so Excel correctly displays non-ASCII
    characters (e.g. accented street/city names) instead of mangling them.
    """
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=LISTING_FIELDNAMES,
                                 extrasaction="ignore")
        writer.writeheader()
        for listing in listings:
            writer.writerow(listing)


def _scrape_band(session, geo_id, max_pages, records_per_page, delay,
                  transaction_type_id, property_type_group_id, sort,
                  seen_mls, price_min=None, price_max=None):
    """Paginate through one search (optionally restricted to a price band)
    until REALTOR.ca's per-search page cap, --pages, or results run out.

    `seen_mls` is a set shared across every band scrape() makes in one run
    -- a listing already collected by an earlier band (or REALTOR.ca's
    price boundaries overlapping slightly) is skipped here rather than
    duplicated. Returns (listings_from_this_band, total_records,
    max_records_cap) -- the latter two describe this band's own search,
    not the overall run.
    """
    band_listings = []
    total_records = None
    max_records_cap = None

    page = 1
    while max_pages is None or page <= max_pages:
        payload = build_payload(
            geo_id, page, records_per_page, transaction_type_id,
            property_type_group_id, sort, price_min=price_min,
            price_max=price_max,
        )

        for attempt in range(3):
            try:
                resp = session.post(SEARCH_URL, data=payload, timeout=20)
                break
            except requests.RequestException as exc:
                print(f"Page {page}: request error ({exc}), "
                      f"retry {attempt + 1}/3...", file=sys.stderr)
                time.sleep(2 * (attempt + 1))
        else:
            print(f"Page {page}: giving up after 3 failed attempts.",
                  file=sys.stderr)
            break

        if resp.status_code != 200:
            print(f"Page {page}: HTTP {resp.status_code} -- stopping. "
                  f"(REALTOR.ca may be rate-limiting/blocking this request.)",
                  file=sys.stderr)
            break

        try:
            data = resp.json()
        except ValueError:
            print(f"Page {page}: response wasn't valid JSON -- stopping.",
                  file=sys.stderr)
            break

        # ErrorCode has appeared in two shapes from this API: a bare value
        # (None / 200 / "200") on older responses, and a nested object like
        # {'Id': 200, 'Description': 'Success - OK', ...} on newer ones.
        # Normalize to just the numeric/None code before checking it, so a
        # successful nested response doesn't get misread as an error.
        error_code = data.get("ErrorCode")
        error_id = error_code.get("Id") if isinstance(error_code, dict) else error_code

        if error_id not in (None, "200", 200):
            print(f"Page {page}: API returned ErrorCode={error_code} "
                  f"-- stopping.", file=sys.stderr)
            break

        paging = data.get("Paging") or {}
        total_records = paging.get("TotalRecords", total_records)
        max_records_cap = paging.get("MaxRecords", max_records_cap)

        results = data.get("Results") or []
        if not results:
            print(f"Page {page}: no more results.", file=sys.stderr)
            break

        new_count = 0
        for raw in results:
            parsed = parse_listing(raw)
            mls = parsed.get("mls_number")
            if mls and mls in seen_mls:
                continue
            if mls:
                seen_mls.add(mls)
            band_listings.append(parsed)
            new_count += 1

        print(f"Page {page}: got {len(results)} listings "
              f"({new_count} new, {len(seen_mls)} total so far).",
              file=sys.stderr)

        records_showing = paging.get("RecordsShowing")
        if max_records_cap and records_showing and records_showing >= max_records_cap:
            print("Reached REALTOR.ca's MaxRecords cap for this "
                  "search/price-band -- stopping this band.", file=sys.stderr)
            break

        total_pages = paging.get("TotalPages")
        if total_pages and page >= total_pages:
            break

        page += 1
        time.sleep(delay)

    return band_listings, total_records, max_records_cap


def probe_count(session, geo_id, transaction_type_id, property_type_group_id,
                 sort, price_min=None, price_max=None):
    """Cheap request (RecordsPerPage=1) that asks the API how many listings
    match a given search/price-band, without fetching the actual results.
    Returns an int, or None if the request failed or the response couldn't
    be read.
    """
    payload = build_payload(
        geo_id=geo_id, page=1, records_per_page=1,
        transaction_type_id=transaction_type_id,
        property_type_group_id=property_type_group_id, sort=sort,
        price_min=price_min, price_max=price_max,
    )
    for attempt in range(2):
        try:
            resp = session.post(SEARCH_URL, data=payload, timeout=20)
            resp.raise_for_status()
            data = resp.json()
            break
        except (requests.RequestException, ValueError):
            time.sleep(1)
    else:
        return None

    error_code = data.get("ErrorCode")
    error_id = error_code.get("Id") if isinstance(error_code, dict) else error_code
    if error_id not in (None, "200", 200):
        return None

    paging = data.get("Paging") or {}
    return paging.get("TotalRecords")


def _price_filter_supported(session, geo_id, transaction_type_id,
                             property_type_group_id, sort, baseline_total):
    """Sanity-check that REALTOR.ca is actually honoring PriceMin/PriceMax
    on this endpoint before relying on it to split up a large search.

    Probes a deliberately tiny $0-$1 price band -- a real listings search
    should return ~0 results for that. If it instead comes back close to
    the full unfiltered total, the filter isn't being applied and
    price-band splitting would silently produce wrong (duplicate-heavy or
    incomplete) results, so callers should skip it instead.
    """
    tiny_count = probe_count(session, geo_id, transaction_type_id,
                              property_type_group_id, sort,
                              price_min=0, price_max=1)
    if tiny_count is None:
        return False
    return tiny_count < max(1, baseline_total * 0.5)


# Used as the top of the initial price range when splitting -- comfortably
# above any realistic residential listing price, so it effectively means
# "no upper bound" without needing a separate open-ended band.
PRICE_BAND_CEILING = 999_000_000
# Stop recursively splitting a band once it's narrower than this (some
# clusters, e.g. a single new-build development, can have hundreds of
# listings at nearly the same price -- no amount of splitting separates
# those, so past this point we just accept REALTOR.ca's cap for that band).
PRICE_BAND_FLOOR = 1000
PRICE_BAND_MAX_DEPTH = 25


def _split_price_bands(session, geo_id, transaction_type_id,
                        property_type_group_id, sort, max_records_cap,
                        price_min=0, price_max=PRICE_BAND_CEILING,
                        probe_delay=0.5, depth=0):
    """Recursively binary-split [price_min, price_max] into price bands,
    each with <= max_records_cap matching listings, by probing counts via
    probe_count(). Returns a list of (price_min, price_max) tuples that
    together cover the full range.
    """
    count = probe_count(session, geo_id, transaction_type_id,
                         property_type_group_id, sort, price_min, price_max)
    time.sleep(probe_delay)

    if not count:
        return []

    if (count <= max_records_cap or depth >= PRICE_BAND_MAX_DEPTH
            or (price_max - price_min) < PRICE_BAND_FLOOR):
        if count > max_records_cap:
            reason = ("max split depth" if depth >= PRICE_BAND_MAX_DEPTH
                       else "minimum band width")
            print(f"Warning: price band ${price_min:,}-${price_max:,} still "
                  f"has {count} listings and can't be split further (hit "
                  f"{reason}) -- only the first {max_records_cap} of these "
                  f"will be scraped.", file=sys.stderr)
        return [(price_min, price_max)]

    mid = (price_min + price_max) // 2
    if mid <= price_min or mid >= price_max:
        return [(price_min, price_max)]

    left = _split_price_bands(session, geo_id, transaction_type_id,
                               property_type_group_id, sort, max_records_cap,
                               price_min, mid, probe_delay, depth + 1)
    right = _split_price_bands(session, geo_id, transaction_type_id,
                                property_type_group_id, sort, max_records_cap,
                                mid + 1, price_max, probe_delay, depth + 1)
    return left + right


def scrape(geo_id, max_pages, records_per_page, delay, transaction_type_id,
           property_type_group_id, sort, split_by_price=True):
    session = requests.Session()
    session.headers.update(DEFAULT_HEADERS)

    # Warm up: fetch the human-facing page first so the session picks up
    # any cookies REALTOR.ca sets before we start hitting the API.
    try:
        session.get(WARMUP_URL, timeout=20)
    except requests.RequestException as exc:
        print(f"Warning: warm-up request failed ({exc}); continuing anyway.",
              file=sys.stderr)

    seen_mls = set()
    all_listings = []
    bands_used = 1

    # Always do one normal, unfiltered pass first. For most searches
    # (total under REALTOR.ca's cap) this is the entire job, identical to
    # how this script always behaved. Listings collected here are kept
    # even if we go on to split by price below -- the shared `seen_mls`
    # set means later bands just skip these instead of re-adding them.
    listings, total_records, max_records_cap = _scrape_band(
        session, geo_id, max_pages, records_per_page, delay,
        transaction_type_id, property_type_group_id, sort, seen_mls,
    )
    all_listings.extend(listings)

    # Only attempt price splitting when: it's enabled, the caller didn't
    # explicitly cap --pages (which signals "I only want N pages, don't
    # fetch more"), and we can actually see that more listings exist than
    # REALTOR.ca will hand back for this search/sort.
    needs_split = (
        split_by_price and max_pages is None
        and total_records is not None and max_records_cap is not None
        and total_records > max_records_cap
    )

    if needs_split:
        print(f"\n{total_records} listings match this search, but "
              f"REALTOR.ca only exposes {max_records_cap} per search/sort "
              f"-- attempting to split by price for more coverage.",
              file=sys.stderr)

        if not _price_filter_supported(session, geo_id, transaction_type_id,
                                        property_type_group_id, sort,
                                        total_records):
            print("Warning: REALTOR.ca doesn't appear to honor price-range "
                  "filtering on this search (or the check itself failed) "
                  "-- skipping price-band splitting. Only the "
                  f"{max_records_cap} listings from the pass above will be "
                  "returned. Re-run with different --geo-id/filters, or "
                  "check the Network tab against build_payload()'s "
                  "PriceMin/PriceMax fields if you want to debug this.",
                  file=sys.stderr)
        else:
            bands = _split_price_bands(
                session, geo_id, transaction_type_id,
                property_type_group_id, sort, max_records_cap,
                probe_delay=max(0.3, delay / 3),
            )
            print(f"Split into {len(bands)} price band(s).", file=sys.stderr)
            bands_used = len(bands) + 1  # +1 for the initial unfiltered pass

            for i, (lo, hi) in enumerate(bands, 1):
                print(f"\n--- Price band {i}/{len(bands)}: "
                      f"${lo:,}-${hi:,} ---", file=sys.stderr)
                band_listings, _tr, _mc = _scrape_band(
                    session, geo_id, max_pages, records_per_page, delay,
                    transaction_type_id, property_type_group_id, sort,
                    seen_mls, price_min=lo, price_max=hi,
                )
                all_listings.extend(band_listings)

    return {
        "listings": all_listings,
        "meta": {
            "scraped_at_utc": datetime.now(timezone.utc).isoformat(),
            "source": WARMUP_URL,
            "geo_id": geo_id,
            "sort": sort,
            "listings_scraped": len(all_listings),
            "total_records_matching_search": total_records,
            "realtor_ca_page_cap": max_records_cap,
            "price_bands_used": bands_used,
        },
    }


def main():
    parser = argparse.ArgumentParser(
        description="Scrape REALTOR.ca listings into a JSON file.")
    parser.add_argument("--geo-id", default="g30_c3nfkdtg",
                         help="REALTOR.ca GeoIds value (default: Calgary, AB).")
    parser.add_argument("--pages", type=int, default=None,
                         help="Max number of pages to fetch (default: until "
                              "REALTOR.ca's cap or results run out).")
    parser.add_argument("--records-per-page", type=int, default=50,
                         help="Listings per request (default: 50).")
    parser.add_argument("--delay", type=float, default=1.5,
                         help="Seconds to wait between requests (default: 1.5).")
    parser.add_argument("--transaction-type-id", type=int, default=2,
                         help="2 = For Sale, as captured from the site "
                              "(default: 2).")
    parser.add_argument("--property-type-group-id", type=int, default=1,
                         help="1 = Residential (default: 1).")
    parser.add_argument("--sort", default="6-D",
                         help="Sort order code (default: '6-D' = Newest).")
    parser.add_argument(
        "--output",
        default=os.path.join(DEFAULT_OUTPUT_DIR, "realtor_listings.json"),
        help="Output JSON file path (default: realtor_listings.json inside "
             f"{DEFAULT_OUTPUT_DIR}).")
    parser.add_argument("--csv-output", default=None,
                         help="Also save listings as CSV at this path "
                              "(default: same name as --output with a "
                              ".csv extension). Pass 'none' to skip CSV "
                              "export.")
    parser.add_argument("--no-price-split", dest="split_by_price",
                         action="store_false",
                         help="Disable automatic price-band splitting. By "
                              "default, if a search matches more listings "
                              "than REALTOR.ca exposes per search/sort "
                              "(currently ~600), this script re-runs the "
                              "search across price bands and merges the "
                              "results to get closer to full coverage. "
                              "Pass this flag to always take just the "
                              "single capped pass instead.")
    parser.set_defaults(split_by_price=True)

    # parse_known_args() (instead of parse_args()) so this still works when
    # run inside Jupyter: Jupyter launches the kernel as
    # "ipykernel_launcher.py -f <connection-file>.json", and that "-f ..."
    # ends up in sys.argv. parse_args() would choke on it as an unrecognized
    # argument ("unrecognized arguments: -f ...json"); parse_known_args()
    # just ignores anything it doesn't recognize and falls back to the
    # defaults above. Works identically from a normal command line too.
    args, _unknown_args = parser.parse_known_args()

    result = scrape(
        geo_id=args.geo_id,
        max_pages=args.pages,
        records_per_page=args.records_per_page,
        delay=args.delay,
        transaction_type_id=args.transaction_type_id,
        property_type_group_id=args.property_type_group_id,
        sort=args.sort,
        split_by_price=args.split_by_price,
    )

    # Resolve to an absolute path before writing. A bare relative filename
    # (the default) already lands in the current working directory -- which
    # in Jupyter is normally the notebook's own folder ("project directory")
    # -- but resolving it explicitly means the path printed below tells you
    # exactly where to look, instead of leaving you to guess.
    output_path = os.path.abspath(os.path.expanduser(args.output))
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f"\nSaved {len(result['listings'])} listings to {output_path}")

    if args.csv_output != "none":
        csv_path = args.csv_output
        if csv_path is None:
            base, _ext = os.path.splitext(args.output)
            csv_path = base + ".csv"
        csv_path = os.path.abspath(os.path.expanduser(csv_path))
        os.makedirs(os.path.dirname(csv_path) or ".", exist_ok=True)
        save_csv(result["listings"], csv_path)
        print(f"Saved {len(result['listings'])} listings to {csv_path} (CSV)")

    print(f"(REALTOR.ca reported {result['meta']['total_records_matching_search']} "
          f"total matching listings; it caps browsable results at "
          f"{result['meta']['realtor_ca_page_cap']} per search/sort. "
          f"Scraped across {result['meta']['price_bands_used']} "
          f"price band(s).)")


if __name__ == "__main__":
    main()

Page 1: got 50 listings (50 new, 50 total so far).
Reached REALTOR.ca's MaxRecords cap for this search/price-band -- stopping this band.

6899 listings match this search, but REALTOR.ca only exposes 600 per search/sort -- attempting to split by price for more coverage.
Split into 10 price band(s).

--- Price band 1/10: $0-$243,896 ---
Page 1: HTTP 403 -- stopping. (REALTOR.ca may be rate-limiting/blocking this request.)

--- Price band 2/10: $243,897-$274,383 ---
Page 1: HTTP 403 -- stopping. (REALTOR.ca may be rate-limiting/blocking this request.)

--- Price band 3/10: $274,384-$304,870 ---
Page 1: HTTP 403 -- stopping. (REALTOR.ca may be rate-limiting/blocking this request.)

--- Price band 4/10: $304,871-$335,357 ---
Page 1: HTTP 403 -- stopping. (REALTOR.ca may be rate-limiting/blocking this request.)

--- Price band 5/10: $335,358-$365,844 ---
Page 1: HTTP 403 -- stopping. (REALTOR.ca may be rate-limiting/blocking this request.)

--- Price band 6/10: $365,845-$426,818 ---
Page 1: 


Saved 50 listings to D:\backup\Downloads\RealtorScrapperProject\realtor_listings.json
Saved 50 listings to D:\backup\Downloads\RealtorScrapperProject\realtor_listings.csv (CSV)
(REALTOR.ca reported 6899 total matching listings; it caps browsable results at 600 per search/sort. Scraped across 11 price band(s).)


Page 1: HTTP 403 -- stopping. (REALTOR.ca may be rate-limiting/blocking this request.)

--- Price band 9/10: $548,768-$579,254 ---
Page 1: HTTP 403 -- stopping. (REALTOR.ca may be rate-limiting/blocking this request.)

--- Price band 10/10: $579,255-$609,741 ---
Page 1: HTTP 403 -- stopping. (REALTOR.ca may be rate-limiting/blocking this request.)
